# Experiment 3a: Zero-shot Topic Classification

Classifies each ticket into one of the 16 `Ticket Subject` categories using `facebook/bart-large-mnli` without any fine-tuning.

Pipeline: load data → BART-MNLI zero-shot → save subject predictions.

In [9]:
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from sklearn.metrics import accuracy_score, f1_score

In [2]:
PROJECT_ROOT = Path.cwd().parents[2]

INPUT_PATH  = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_input.csv'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'Experiment_3_BERTMNLI'

OUT_PATH = RESULTS_DIR / 'Experiment_3a_subject_predictions.csv'

In [3]:
df = pd.read_csv(INPUT_PATH)
SUBJECT_LABELS = sorted(df['Ticket Subject'].unique().tolist())

print(f'Labels:', SUBJECT_LABELS)

Labels: ['Account access', 'Battery life', 'Cancellation request', 'Data loss', 'Delivery problem', 'Display issue', 'Hardware issue', 'Installation support', 'Network problem', 'Payment issue', 'Peripheral compatibility', 'Product compatibility', 'Product recommendation', 'Product setup', 'Refund request', 'Software bug']


In [4]:
DEVICE = 0

classifier = pipeline(
    task='zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=DEVICE
)

Device set to use cuda:0


When the BART tokenizer encounters a newline character (\r\n), it inserts an extra eos token in the middle of the sequence.

If the number of eos tokens is inconsistent across texts within the same batch, the BART classification header will throw an error.  
Note: do not change unless absolutely necessary the `batch_size` both in 03a and 03b.

In [5]:
# Use description_only (Ticket Description only, without Ticket Subject prefix)
# to avoid label leakage into the classification input.
clean_texts = (
    df['description_only']
    .astype(str)
    .str[:512]
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [6]:
ds = Dataset.from_dict({'text': clean_texts.tolist()})

pred_subjects, pred_confs = [], []
for result in classifier(KeyDataset(ds, 'text'), candidate_labels=SUBJECT_LABELS, truncation=True, max_length=256, batch_size=8):
    pred_subjects.append(result['labels'][0])
    pred_confs.append(round(result['scores'][0], 4))

In [7]:
out = pd.DataFrame({
    'ticket_id': df['ticket_id'].values,
    'text': clean_texts.values,
    'pred_subject': pred_subjects,
    'pred_subject_conf': pred_confs,
    'true_subject': df['Ticket Subject'].values
})
out.to_csv(OUT_PATH, index=False, encoding='utf-8')

out.head()

,ticket_id,text,pred_subject,pred_subject_conf,true_subject
0,TICKET_0,I'm having an issue with the . Please assist. ...,Display issue,0.2787,Product setup
1,TICKET_1,I'm having an issue with the . Please assist. ...,Product compatibility,0.2560,Peripheral compatibility
2,TICKET_2,I'm facing a problem with my . The is not turn...,Display issue,0.1980,Network problem
3,TICKET_3,I'm having an issue with the . Please assist. ...,Display issue,0.1794,Account access
4,TICKET_4,I'm having an issue with the . Please assist. ...,Battery life,0.4403,Data loss


## Conclusion & Exploration

The zero-shot classification results appear to refute our hypothesis, with BART-MNLI achieving an accuracy of 6.2% and a macro F1 of 0.047 across 16 Ticket Subject categories which equivalents to random chance.

However, this outcome should not be interpreted as a model capability boundary. The synthetic dataset generates all ticket descriptions from a fixed template ("I'm having an issue with the {product_purchased}. Please assist."), producing near-identical text across all 16 subject categories once placeholders are removed. No model — zero-shot or supervised — can distinguish classes from descriptions that carry no class-specific lexical signal. The failure is therefore attributable to data homogeneity, not to the semantic capacity of BART-MNLI.

We apply the same zero-shot approach to the larger real-world dataset using its `subject` field, where customer descriptions are diverse and class-specific.

## Zero-shot Tag Classification — Large Dataset in TF-IDF and BART-MNLI

## Hypothesis

H1 - Real customer ticket text contains enough semantic cues to recover both ground-truth tags (tag_1, tag_2) better than chance with zero-shot NLI.
Reason: unlike the templated small dataset, this corpus has diverse lexical and contextual signals.

H2 - TF-IDF candidate recall before BART-MNLI reranking can preserve most true tags while reducing inference cost.
Reason: lexical overlap narrows candidate space, and BART-MNLI remains the final semantic judge.

## Pipeline

**load large data -> build tag pool -> TF-IDF recall Top-M candidates -> BART-MNLI rerank -> evaluate order-invariant Top-2 tags.**

In [16]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

LARGE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'large_dataset_preprocessed.csv'
OUT_LARGE_PRED_PATH = RESULTS_DIR / 'Experiment_3a_large_tfidf_mnli_predictions.csv'

# Speed/coverage controls
TOPK_TAG_POOL = 80      # candidate tag universe size (set to None for all tags)
CANDIDATE_TOP_M = 20    # TF-IDF recall size before MNLI reranking
EVAL_TOP_K = 2          # final predicted tag count
MAX_ROWS_FOR_EXPERIMENT = 4000   # set to None for full subset
MAX_TEXTS_PER_TAG = 128

df_large = pd.read_csv(LARGE_PATH)

In [17]:
# Keep rows with non-empty text and both tags available
for c in ['text_cleaned', 'tag_1', 'tag_2']:
    df_large[c] = df_large[c].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    df_large.loc[df_large[c].isin(['', 'nan', 'NaN', 'None']), c] = pd.NA

df_large = df_large.dropna(subset=['text_cleaned', 'tag_1', 'tag_2']).copy()

# Build a tag pool from combined tag_1/tag_2 frequencies
tag_counts = pd.concat([df_large['tag_1'], df_large['tag_2']]).value_counts()
if TOPK_TAG_POOL is None:
    TAG_POOL = sorted(tag_counts.index.tolist())
else:
    TAG_POOL = tag_counts.head(TOPK_TAG_POOL).index.tolist()

In [18]:
# Keep rows where both true tags are inside the selected pool
work = df_large[df_large['tag_1'].isin(TAG_POOL) & df_large['tag_2'].isin(TAG_POOL)].copy()
if MAX_ROWS_FOR_EXPERIMENT is not None and len(work) > MAX_ROWS_FOR_EXPERIMENT:
    work = work.sample(n=MAX_ROWS_FOR_EXPERIMENT, random_state=42).reset_index(drop=True)

In [19]:
# Build one prototype document per tag
tag_docs = []
for tag in TAG_POOL:
    texts_for_tag = work.loc[(work['tag_1'] == tag) | (work['tag_2'] == tag), 'text_cleaned']
    texts_for_tag = texts_for_tag.drop_duplicates().head(MAX_TEXTS_PER_TAG)
    tag_docs.append(' '.join(texts_for_tag.tolist()))

vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
tag_matrix = vectorizer.fit_transform(tag_docs)

pred_tag1, pred_tag2 = [''] * len(work), [''] * len(work)
pred_tag1_conf, pred_tag2_conf = [0.0] * len(work), [0.0] * len(work)
candidate_trace = [''] * len(work)

texts = work['text_cleaned'].tolist()
top_m = min(CANDIDATE_TOP_M, len(TAG_POOL))

# 1) TF-IDF recall for each text
candidate_groups = {}  # key: candidate tuple, value: list of row indices
for i, text in enumerate(texts):
    q = vectorizer.transform([text])
    sims = cosine_similarity(q, tag_matrix).ravel()

    candidate_idx = np.argpartition(-sims, top_m - 1)[:top_m]
    candidate_idx = candidate_idx[np.argsort(-sims[candidate_idx])]
    candidate_labels = [TAG_POOL[j] for j in candidate_idx]

    candidate_trace[i] = ' | '.join(candidate_labels)
    key = tuple(candidate_labels)
    candidate_groups.setdefault(key, []).append(i)

# 2) Batch MNLI by candidate group to avoid fully sequential GPU calls
batch_size_mnli = 8
processed = 0
for key, idxs in candidate_groups.items():
    group_texts = [texts[j] for j in idxs]
    ds_group = Dataset.from_dict({'text': group_texts})
    outputs = classifier(
        KeyDataset(ds_group, 'text'),
        candidate_labels=list(key),
        truncation=True,
        max_length=256,
        batch_size=batch_size_mnli,
        multi_label=True
    )

    for local_k, result in enumerate(outputs):
        labels = result['labels'][:EVAL_TOP_K]
        scores = result['scores'][:EVAL_TOP_K]

        if len(labels) < 2:
            labels = labels + [''] * (2 - len(labels))
            scores = scores + [0.0] * (2 - len(scores))

        row_idx = idxs[local_k]
        pred_tag1[row_idx] = labels[0]
        pred_tag2[row_idx] = labels[1]
        pred_tag1_conf[row_idx] = round(float(scores[0]), 4)
        pred_tag2_conf[row_idx] = round(float(scores[1]), 4)

    processed += len(idxs)
    if processed % 200 == 0 or processed == len(texts):
        print(f'Processed {processed}/{len(texts)} rows')

print('TF-IDF recall + grouped-batch MNLI rerank complete.')

Processed 200/4000 rows
Processed 400/4000 rows
Processed 600/4000 rows
Processed 800/4000 rows
Processed 1000/4000 rows
Processed 1200/4000 rows
Processed 1400/4000 rows
Processed 1600/4000 rows
Processed 1800/4000 rows
Processed 2000/4000 rows
Processed 2200/4000 rows
Processed 2400/4000 rows
Processed 2600/4000 rows
Processed 2800/4000 rows
Processed 3000/4000 rows
Processed 3200/4000 rows
Processed 3400/4000 rows
Processed 3600/4000 rows
Processed 3800/4000 rows
Processed 4000/4000 rows
TF-IDF recall + grouped-batch MNLI rerank complete.


### Evaluation Metrics：

- Exact Match@2: predicted tag set exactly matches the two ground-truth tags (order ignored).
- Hit@2: at least one ground-truth tag appears in the predicted top-2 set.
- Jaccard: average overlap ratio between predicted and true tag sets.
- Micro-F1: global multi-label F1, weighted by overall label frequency.
- Macro-F1: per-label F1 average, sensitive to tail-label performance.

These metrics separate strict correctness (Exact Match@2) from partial relevance (Hit@2/Jaccard).

In [22]:
# Order-invariant evaluation for two-tag prediction
eval_df = pd.DataFrame({
    'text': work['text_cleaned'].values,
    'true_tag_1': work['tag_1'].values,
    'true_tag_2': work['tag_2'].values,
    'pred_tag_1': pred_tag1,
    'pred_tag_2': pred_tag2,
    'pred_tag_1_conf': pred_tag1_conf,
    'pred_tag_2_conf': pred_tag2_conf,
    'candidate_labels_top_m': candidate_trace
})

true_sets = [set([a, b]) for a, b in zip(eval_df['true_tag_1'], eval_df['true_tag_2'])]
pred_sets = [set([a, b]) for a, b in zip(eval_df['pred_tag_1'], eval_df['pred_tag_2'])]

exact_match_at2 = np.mean([t == p for t, p in zip(true_sets, pred_sets)])
hit_at2 = np.mean([len(t & p) > 0 for t, p in zip(true_sets, pred_sets)])
jaccard = np.mean([len(t & p) / len(t | p) if len(t | p) > 0 else 0.0 for t, p in zip(true_sets, pred_sets)])

mlb = MultiLabelBinarizer(classes=TAG_POOL)
y_true = mlb.fit_transform([list(s) for s in true_sets])
y_pred = mlb.transform([list(s) for s in pred_sets])

micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print(f'Exact Match@2: {exact_match_at2:.4f}')
print(f'Hit@2: {hit_at2:.4f}')
print(f'Jaccard: {jaccard:.4f}')
print(f'Micro-F1: {micro_f1:.4f}')
print(f'Macro-F1: {macro_f1:.4f}')

Exact Match@2: 0.0255
Hit@2: 0.3902
Jaccard: 0.1471
Micro-F1: 0.2079
Macro-F1: 0.1807


In [ ]:
# TF-IDF Top-M candidate coverage vs ground truth + random baselines
from math import comb

candidate_sets = [set(x.split(' | ')) if isinstance(x, str) and x.strip() else set() for x in eval_df['candidate_labels_top_m']]
true_sets_eval = [set([a, b]) for a, b in zip(eval_df['true_tag_1'], eval_df['true_tag_2'])]

tfidf_any_hit = np.mean([len(c & t) >= 1 for c, t in zip(candidate_sets, true_sets_eval)])
tfidf_both_hit = np.mean([len(c & t) == 2 for c, t in zip(candidate_sets, true_sets_eval)])
tfidf_miss_both = 1.0 - tfidf_any_hit

# Uniform random baseline for final Top-2 prediction over TAG_POOL
K = len(TAG_POOL)
random_exact_match_at2 = 1 / comb(K, 2)
random_hit_at2 = 1 - (comb(K - 2, 2) / comb(K, 2))

print(f'TF-IDF candidate contains both true tags: {tfidf_both_hit:.4f}')
print(f'Random Exact Match@2 baseline (K={K}): {random_exact_match_at2:.6f}')
print(f'Random Hit@2 baseline (K={K}): {random_hit_at2:.4f}')

TF-IDF candidate contains >=1 true tag: 0.9995
TF-IDF candidate contains both true tags: 0.9852
TF-IDF candidate misses both true tags: 0.0005
Random Exact Match@2 baseline (K=80): 0.000316
Random Hit@2 baseline (K=80): 0.0497


In [24]:
eval_df.to_csv(OUT_LARGE_PRED_PATH, index=False, encoding='utf-8')

## Final Remarks

Compared with random baselines, the model is not guessing blindly: Top-2 metrics are consistently above chance, and TF-IDF Top-20 candidates recover ground-truth tags at very high rates. This confirms that textual semantics is informative for tag assignment.

However, strict two-tag correctness remains low, and macro-level performance is still limited. A likely cause is domain mismatch between generic pretrained MNLI decision boundaries and this task-specific tag taxonomy.

Therefore, the next step is supervised classifier fine-tuning a task-specific multi-label head, while retaining TF-IDF candidate recall as an efficiency layer.